# Batch Inference Pipeline with MLflow Model

## Topic Goal

This notebook properly implements a **batch inference pipeline** using an MLflow model.



1. Train a model.
2. Evaluate the model.
3. Infer model signature.
4. Start one MLflow run.
5. Log parameters, metrics, signature, input example, and model.
6. Perform batch inference inside the **same run**.
7. Save batch predictions to CSV.
8. Log the batch prediction CSV as an MLflow artifact.
9. Create `model_uri` from the same active run.
10. Register the model using the same `model_uri`.

This is a realistic batch scoring pattern used in production MLOps.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, and utilities required for training, scoring, model signature, artifact logging, and registry.


In [ ]:
# Import os to create folders and clean MLflow environment variables.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking, model logging, and registry.
import mlflow

# Import MLflow scikit-learn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature for capturing model input and output schema.
from mlflow.models.signature import infer_signature

# Import pandas for reading and writing tabular data.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import train_test_split for splitting training data.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for preprocessing different column types.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical encoding.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical scaling.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Folders

This notebook uses local file-based MLflow tracking.

It creates:

- `mlruns/` for MLflow tracking data
- `outputs/` for batch prediction CSV
- `artifacts/` for additional supporting files


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_08_batch_inference_pipeline"

# Define run name.
RUN_NAME = "08_batch_inference_pipeline_same_run_usecase"

# Remove inherited MLflow Project experiment variable if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run variable if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for batch prediction files.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for additional supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure MLflow to use local tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define training dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Define current/batch scoring dataset path.
CURRENT_DATA_PATH = "datasets/customer_churn_current_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Training dataset:", DATA_PATH)
print("Batch scoring dataset:", CURRENT_DATA_PATH)
print("Tracking URI:", mlflow.get_tracking_uri())


## 3. Load Training Dataset

We load the baseline customer churn dataset for model training.

The target column is:

```text
churn
```


In [ ]:
# Load baseline training dataset.
df = pd.read_csv(DATA_PATH)

# Display first five records.
display(df.head())

# Define target column.
target_column = "churn"

# Create feature matrix by dropping customer ID and target.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature groups.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print data split information.
print("Training dataset shape:", df.shape)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 4. Load Batch Scoring Dataset

In a real batch inference pipeline, we usually score a **new/current dataset**, not the same training dataset.

This notebook uses:

```text
datasets/customer_churn_current_500.csv
```

If that file is not available, the notebook will fall back to the baseline dataset.


In [ ]:
# Check whether current batch scoring dataset exists.
if os.path.exists(CURRENT_DATA_PATH):
    # Load current dataset for batch scoring.
    batch_df = pd.read_csv(CURRENT_DATA_PATH)

    # Print source used.
    print("Loaded current batch scoring dataset:", CURRENT_DATA_PATH)
else:
    # Fall back to baseline dataset when current dataset is unavailable.
    batch_df = pd.read_csv(DATA_PATH)

    # Print fallback message.
    print("Current dataset not found. Falling back to:", DATA_PATH)

# Display sample records from batch dataset.
display(batch_df.head())

# Prepare batch input by dropping customer ID and target if target exists.
batch_input = batch_df.drop(columns=["customer_id", "churn"], errors="ignore")

# Print batch input shape.
print("Batch input shape:", batch_input.shape)


## 5. Train and Evaluate Model

We train a complete scikit-learn pipeline containing preprocessing and the model.

This is important because the logged model can accept raw categorical and numerical input during batch scoring.


In [ ]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input data.
    ("preprocessor", preprocessor),

    # Train Random Forest model.
    ("model", model),
])

# Train pipeline.
pipeline.fit(X_train, y_train)

# Generate predictions on test data.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report.
report = classification_report(y_test, predictions, zero_division=0)

# Print metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(report)


## 6. Infer Signature

The model signature documents the expected input and output schema.

This helps deployment and batch scoring teams understand the required input format.


In [ ]:
# Create input example for model signature.
input_example = X_test.head(5)

# Generate sample predictions for the input example.
sample_predictions = pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 7. Same-Run Model Logging, Batch Inference, Artifact Logging, and Registry

This is the corrected implementation.

Inside the **same MLflow run**, we:

1. Log parameters.
2. Log metrics.
3. Log model with signature and input example.
4. Run batch inference.
5. Save batch prediction CSV.
6. Log batch prediction CSV as artifact.
7. Create `model_uri`.

After the run closes, we register the model using that same `model_uri`.


In [ ]:
# Start the SAME MLflow run for training metrics, model, signature, batch output, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log training dataset path.
    mlflow.log_param("training_dataset", DATA_PATH)

    # Log batch scoring dataset path.
    mlflow.log_param("batch_scoring_dataset", CURRENT_DATA_PATH if os.path.exists(CURRENT_DATA_PATH) else DATA_PATH)

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Save classification report locally.
    with open("outputs/classification_report.txt", "w") as file:
        file.write(report)

    # Log classification report artifact.
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log the trained model with signature and input example.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from the same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

    # Load the same logged model using MLflow pyfunc.
    loaded_model = mlflow.pyfunc.load_model(model_uri)

    # Generate batch predictions using the logged model.
    batch_predictions = loaded_model.predict(batch_input)

    # Create batch output dataframe.
    batch_output = batch_df.copy()

    # Add prediction column.
    batch_output["predicted_churn"] = batch_predictions

    # Save batch prediction output.
    batch_output_path = "outputs/batch_churn_predictions.csv"
    batch_output.to_csv(batch_output_path, index=False)

    # Log batch prediction output as MLflow artifact.
    mlflow.log_artifact(batch_output_path)

    # Log number of scored records.
    mlflow.log_metric("batch_records_scored", len(batch_output))

# Register model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print same-run model URI.
print("Same-run model URI:", model_uri)

# Print registered model details.
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)

# Display batch output sample.
display(batch_output.head())


## 8. Verify Batch Output

This section verifies that the batch prediction file was created and contains predictions.


In [ ]:
# Read the saved batch prediction file.
saved_batch_output = pd.read_csv("outputs/batch_churn_predictions.csv")

# Display first few rows.
display(saved_batch_output.head())

# Print output shape.
print("Saved batch output shape:", saved_batch_output.shape)

# Print output file path.
print("Batch prediction file:", "outputs/batch_churn_predictions.csv")


## 9. Trainer Notes



- Batch inference is used when we want to score many records at once.
- Unlike real-time serving, batch scoring usually runs on a schedule.
- The model is trained and logged with signature.
- The same logged model is loaded using `mlflow.pyfunc.load_model`.
- The batch dataset is scored and saved as a CSV output.
- The batch prediction CSV is logged as an MLflow artifact.
- This makes the run auditable because the run contains the model, metrics, signature, and prediction output.
- The registered model source points to the same run that produced the batch scoring output.
